# Dataset quality control

Compute library sizes, detected genes, mitochondrial fractions, and QC summary tables for the manuscript count matrices. Input files are read without modification; thresholds flag cells for reporting and do not filter the data.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "notebooks" / "figures" / "output" / "qc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Output directory: {OUT_DIR}")

## Dataset configuration

`DATASETS` specifies the input matrices. The text summary includes datasets marked `source_type="real"`; tables and plots also include the simulated datasets.


In [ ]:
DATASETS = [
    {"label": "CORTEX", "path": DATA_DIR / "cortex.h5ad", "source_type": "real", "role": "benchmark; imputation"},
    {"label": "HCA Nuclei", "path": DATA_DIR / "hca_nuclei.h5ad", "source_type": "real", "role": "batch benchmark"},
    {"label": "PBMC CITE-seq", "path": DATA_DIR / "pbmc_cite_seq.h5ad", "source_type": "real", "role": "joint cell-gene clustering; benchmark"},
    {"label": "IFN-beta stim", "path": DATA_DIR / "kang_2018.h5ad", "source_type": "real", "role": "IFN-beta perturbation"},
    {"label": "NeurIPS BMMC GEX", "path": DATA_DIR / "neurips_cite_gex.h5ad", "source_type": "real", "role": "benchmark; zero-inflation diagnostics"},
    {"label": "NeurIPS stem/progenitor", "path": DATA_DIR / "neurips_cite_gex_stem_cells.h5ad", "source_type": "real", "role": "ultra-low-dimensional comparison"},
    {"label": "GR control", "path": DATA_DIR / "GR_00.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "GR 1h", "path": DATA_DIR / "GR_01.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "GR 2h", "path": DATA_DIR / "GR_02.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "GR 4h", "path": DATA_DIR / "GR_04.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "GR 8h", "path": DATA_DIR / "GR_08.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "GR 18h", "path": DATA_DIR / "GR_18.h5ad", "source_type": "real", "role": "GR time course"},
    {"label": "TCR stim", "path": DATA_DIR / "tcr_stim_data.h5ad", "source_type": "real", "role": "TCR perturbation"},
    {"label": "SIM1", "path": DATA_DIR / "sim1_1_norm.h5ad", "source_type": "synthetic", "role": "Splatter simulation benchmark"},
    {"label": "SIM2", "path": DATA_DIR / "sim2_norm.h5ad", "source_type": "synthetic", "role": "Splatter simulation benchmark"},
]

missing = [d for d in DATASETS if not Path(d["path"]).exists()]
if missing:
    raise FileNotFoundError("Missing configured datasets: " + ", ".join(str(d["path"]) for d in missing))

pd.DataFrame(DATASETS)

## QC helpers

Automatic matrix selection checks count layers first, then `.raw.X`, then `.X`. Matrices are processed in cell chunks.


In [ ]:
COUNT_LAYER_CANDIDATES = (
    "counts",
    "count",
    "raw_counts",
    "raw_count",
    "umi_counts",
    "UMI_counts",
)
MT_PREFIXES = ("MT-", "mt-", "Mt-")
GENE_SYMBOL_COLUMNS = ("gene_symbols", "gene_symbol", "symbol", "gene_name", "features")
COMMON_OBS_COLUMNS = (
    "batch",
    "Batch",
    "sample",
    "donor",
    "condition",
    "label",
    "time",
    "time_hr",
    "cell_type",
    "celltype",
    "celltype.l1",
    "Group",
    "Sub",
)


def choose_matrix(adata: ad.AnnData, requested: str = "auto") -> tuple[str, Any, pd.Index]:
    requested = requested.strip()

    if requested == "X":
        return "X", adata.X, adata.var_names

    if requested == "raw":
        if adata.raw is None:
            raise ValueError("Requested raw matrix, but adata.raw is absent.")
        return "raw.X", adata.raw.X, adata.raw.var_names

    if requested.startswith("layer:"):
        layer_name = requested.split(":", 1)[1]
        if layer_name not in adata.layers:
            raise ValueError(f"Requested layer is absent: {layer_name}")
        return f"layers['{layer_name}']", adata.layers[layer_name], adata.var_names

    if requested != "auto":
        raise ValueError("requested must be 'auto', 'X', 'raw', or 'layer:<name>'.")

    for layer_name in COUNT_LAYER_CANDIDATES:
        if layer_name in adata.layers:
            return f"layers['{layer_name}']", adata.layers[layer_name], adata.var_names

    if adata.raw is not None:
        return "raw.X", adata.raw.X, adata.raw.var_names

    return "X", adata.X, adata.var_names


def gene_name_vector(adata: ad.AnnData, var_names: pd.Index) -> pd.Series:
    for column in GENE_SYMBOL_COLUMNS:
        if column in adata.var and len(adata.var[column]) == len(var_names):
            return adata.var[column].astype(str).reset_index(drop=True)
    return pd.Series(var_names.astype(str), index=range(len(var_names)))


def mt_mask(adata: ad.AnnData, var_names: pd.Index) -> np.ndarray:
    names = gene_name_vector(adata, var_names)
    return names.astype(str).str.startswith(MT_PREFIXES).to_numpy(dtype=bool)


def as_array_1d(x: Any) -> np.ndarray:
    return np.asarray(x).ravel()


def summarize_vector(values: np.ndarray, prefix: str) -> dict[str, float]:
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {f"{prefix}_{k}": np.nan for k in ["min", "p01", "p05", "median", "mean", "p95", "p99", "max"]}
    qs = np.quantile(finite, [0, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0])
    return {
        f"{prefix}_min": qs[0],
        f"{prefix}_p01": qs[1],
        f"{prefix}_p05": qs[2],
        f"{prefix}_median": qs[3],
        f"{prefix}_mean": float(np.mean(finite)),
        f"{prefix}_p95": qs[4],
        f"{prefix}_p99": qs[5],
        f"{prefix}_max": qs[6],
    }


def summarize_obs_categories(adata: ad.AnnData, max_levels: int = 10) -> pd.DataFrame:
    rows = []
    columns = [c for c in COMMON_OBS_COLUMNS if c in adata.obs]
    for column in adata.obs.columns:
        if column in columns:
            continue
        nunique = adata.obs[column].nunique(dropna=True)
        if 1 < nunique <= max_levels:
            columns.append(column)

    for column in columns[:14]:
        series = adata.obs[column]
        counts = series.astype("string").fillna("<NA>").value_counts(dropna=False).head(max_levels)
        rows.append({
            "obs_column": column,
            "unique": int(series.nunique(dropna=True)),
            "top_levels": "; ".join(f"{idx}: {int(val):,}" for idx, val in counts.items()),
        })
    return pd.DataFrame(rows)


def compute_qc_for_dataset(dataset: dict[str, Any], chunk_size: int = 4096, matrix: str = "auto") -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    path = Path(dataset["path"])
    adata = ad.read_h5ad(path, backed="r")
    try:
        matrix_name, X, var_names = choose_matrix(adata, requested=matrix)
        mt = mt_mask(adata, var_names)
        n_obs, n_vars = X.shape

        counts_per_cell = np.zeros(n_obs, dtype=float)
        genes_per_cell = np.zeros(n_obs, dtype=float)
        mt_counts_per_cell = np.zeros(n_obs, dtype=float)
        counts_per_gene = np.zeros(n_vars, dtype=float)
        cells_per_gene = np.zeros(n_vars, dtype=float)
        nnz = 0
        min_value = np.inf
        max_value = -np.inf
        noninteger_checked = 0
        noninteger_seen = 0

        for start in range(0, n_obs, chunk_size):
            end = min(start + chunk_size, n_obs)
            block = X[start:end, :]

            if sparse.issparse(block):
                block = block.tocsr()
                row_sums = as_array_1d(block.sum(axis=1))
                row_nnz = as_array_1d(block.getnnz(axis=1))
                col_sums = as_array_1d(block.sum(axis=0))
                col_nnz = as_array_1d(block.getnnz(axis=0))
                block_min = float(block.min()) if block.nnz else 0.0
                block_max = float(block.max()) if block.nnz else 0.0
                nnz += int(block.nnz)
                mt_counts = as_array_1d(block[:, mt].sum(axis=1)) if mt.any() else np.zeros(end - start, dtype=float)
                sample_values = block.data[:10000]
            else:
                block = np.asarray(block)
                row_sums = block.sum(axis=1)
                row_nnz = np.count_nonzero(block > 0, axis=1)
                col_sums = block.sum(axis=0)
                col_nnz = np.count_nonzero(block > 0, axis=0)
                block_min = float(np.nanmin(block)) if block.size else np.nan
                block_max = float(np.nanmax(block)) if block.size else np.nan
                nnz += int(np.count_nonzero(block))
                mt_counts = block[:, mt].sum(axis=1) if mt.any() else np.zeros(end - start, dtype=float)
                sample_values = block.ravel()[:10000]

            counts_per_cell[start:end] = row_sums
            genes_per_cell[start:end] = row_nnz
            mt_counts_per_cell[start:end] = mt_counts
            counts_per_gene += col_sums
            cells_per_gene += col_nnz
            min_value = min(min_value, block_min)
            max_value = max(max_value, block_max)

            if len(sample_values):
                checked = np.asarray(sample_values, dtype=float)
                checked = checked[np.isfinite(checked)]
                checked = checked[checked != 0]
                checked = checked[:2000]
                noninteger_checked += checked.size
                noninteger_seen += int(np.sum(np.abs(checked - np.round(checked)) > 1e-6))

        pct_mt = np.divide(
            mt_counts_per_cell,
            counts_per_cell,
            out=np.full_like(counts_per_cell, np.nan, dtype=float),
            where=counts_per_cell != 0,
        ) * 100

        obs_qc = pd.DataFrame({
            "dataset": dataset["label"],
            "counts_per_cell": counts_per_cell,
            "genes_per_cell": genes_per_cell,
            "pct_mt": pct_mt,
        })
        var_qc = pd.DataFrame({
            "dataset": dataset["label"],
            "gene": np.asarray(var_names.astype(str)),
            "counts_per_gene": counts_per_gene,
            "cells_per_gene": cells_per_gene,
            "pct_cells": cells_per_gene / n_obs * 100 if n_obs else np.nan,
            "is_mt": mt,
        })

        summary = {
            "dataset": dataset["label"],
            "file": path.name,
            "path": str(path.relative_to(ROOT)),
            "source_type": dataset["source_type"],
            "role": dataset["role"],
            "n_cells": int(n_obs),
            "n_genes": int(n_vars),
            "matrix": matrix_name,
            "matrix_density": nnz / float(n_obs * n_vars),
            "matrix_min": min_value,
            "matrix_max": max_value,
            "noninteger_sample_fraction": noninteger_seen / noninteger_checked if noninteger_checked else 0.0,
            "mt_genes_detected": int(mt.sum()),
            "obs_columns": int(len(adata.obs.columns)),
            "var_columns": int(len(adata.var.columns)),
            "layers": ", ".join(map(str, adata.layers.keys())) or "none",
            "obsm": ", ".join(map(str, adata.obsm.keys())) or "none",
            "zero_count_cells": int(np.sum(counts_per_cell == 0)),
            "zero_detected_gene_cells": int(np.sum(genes_per_cell == 0)),
            "zero_count_genes": int(np.sum(counts_per_gene == 0)),
            "zero_detected_cell_genes": int(np.sum(cells_per_gene == 0)),
            "cells_lt_200_genes": int(np.sum(genes_per_cell < 200)),
            "cells_lt_500_genes": int(np.sum(genes_per_cell < 500)),
            "cells_mt_gt_10pct": int(np.nansum(pct_mt > 10)) if mt.any() else np.nan,
            "cells_mt_gt_20pct": int(np.nansum(pct_mt > 20)) if mt.any() else np.nan,
        }
        summary.update(summarize_vector(counts_per_cell, "counts_per_cell"))
        summary.update(summarize_vector(genes_per_cell, "genes_per_cell"))
        summary.update(summarize_vector(pct_mt, "pct_mt"))
        summary.update(summarize_vector(counts_per_gene, "counts_per_gene"))
        summary.update(summarize_vector(cells_per_gene, "cells_per_gene"))

        obs_categories = summarize_obs_categories(adata)
        obs_categories.insert(0, "dataset", dataset["label"])
        summary["obs_category_summary"] = obs_categories
        return summary, obs_qc, var_qc
    finally:
        adata.file.close()

## Run QC

This cell can take a few minutes because it scans each configured count matrix. The input files are opened with `backed='r'` and processed in chunks.

In [ ]:
summaries = []
obs_tables = []
var_tables = []
obs_category_tables = []

for i, dataset in enumerate(DATASETS, start=1):
    print(f"[{i}/{len(DATASETS)}] {dataset['label']} - {Path(dataset['path']).name}")
    summary, obs_qc, var_qc = compute_qc_for_dataset(dataset, chunk_size=4096)
    obs_category_tables.append(summary.pop("obs_category_summary"))
    summaries.append(summary)
    obs_tables.append(obs_qc)
    var_tables.append(var_qc)

qc_summary = pd.DataFrame(summaries)
obs_qc = pd.concat(obs_tables, ignore_index=True)
var_qc = pd.concat(var_tables, ignore_index=True)
obs_category_summary = pd.concat(obs_category_tables, ignore_index=True)

qc_summary.to_csv(OUT_DIR / "paper_dataset_qc_summary.csv", index=False)
obs_category_summary.to_csv(OUT_DIR / "paper_dataset_obs_categories.csv", index=False)

print(f"Wrote {OUT_DIR / 'paper_dataset_qc_summary.csv'}")
print(f"Wrote {OUT_DIR / 'paper_dataset_obs_categories.csv'}")

## Summary Table

Columns ending in `p05`, `median`, and `p95` summarize per-cell or per-gene distributions. The flag columns report common QC sensitivity thresholds but do not apply filters.

In [ ]:
display_cols = [
    "dataset", "source_type", "n_cells", "n_genes", "matrix", "matrix_density", "mt_genes_detected",
    "counts_per_cell_p05", "counts_per_cell_median", "counts_per_cell_p95",
    "genes_per_cell_p05", "genes_per_cell_median", "genes_per_cell_p95",
    "pct_mt_median", "pct_mt_p95", "zero_count_cells", "zero_detected_gene_cells",
    "zero_detected_cell_genes", "cells_lt_200_genes", "cells_lt_500_genes", "cells_mt_gt_10pct", "cells_mt_gt_20pct",
]

summary_display = qc_summary[display_cols].copy()
for col in summary_display.select_dtypes(include=["float"]).columns:
    summary_display[col] = summary_display[col].round(3)

summary_display

In [ ]:
def add_percent_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["zero_count_cells", "zero_detected_gene_cells", "cells_lt_200_genes", "cells_lt_500_genes", "cells_mt_gt_10pct", "cells_mt_gt_20pct"]:
        out[f"{col}_pct"] = out[col] / out["n_cells"] * 100
    out["zero_detected_cell_genes_pct"] = out["zero_detected_cell_genes"] / out["n_genes"] * 100
    return out

flag_display = add_percent_columns(qc_summary)[[
    "dataset", "source_type", "zero_count_cells_pct", "zero_detected_gene_cells_pct",
    "cells_lt_200_genes_pct", "cells_lt_500_genes_pct", "cells_mt_gt_10pct_pct", "cells_mt_gt_20pct_pct",
    "zero_detected_cell_genes", "zero_detected_cell_genes_pct",
]].copy()

for col in flag_display.select_dtypes(include=["float"]).columns:
    flag_display[col] = flag_display[col].round(3)

flag_display

## Metadata Columns

This table records the main sample/cell annotation columns that may explain QC structure by donor, batch, condition, cell type, or time point.

In [ ]:
obs_category_summary

## QC Plots

The plots use sampled cells for speed and readability. The full summary tables above are computed from all cells and genes.

In [ ]:
rng = np.random.default_rng(42)
order = qc_summary["dataset"].tolist()
plot_obs = pd.concat(
    [obs_qc.loc[obs_qc["dataset"].eq(d)].sample(n=min((obs_qc["dataset"] == d).sum(), 6000), random_state=42) for d in order],
    ignore_index=True,
)
plot_obs["log10_counts_per_cell"] = np.log10(plot_obs["counts_per_cell"].clip(lower=1))
plot_obs["log10_genes_per_cell"] = np.log10(plot_obs["genes_per_cell"].clip(lower=1))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for ax, col, ylabel in [
    (axes[0], "log10_counts_per_cell", "log10 counts per cell"),
    (axes[1], "log10_genes_per_cell", "log10 detected genes per cell"),
    (axes[2], "pct_mt", "mitochondrial counts (%)"),
]:
    data = [plot_obs.loc[plot_obs["dataset"].eq(d), col].dropna().to_numpy() for d in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)

fig.savefig(OUT_DIR / "paper_dataset_qc_boxplots.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
flag_plot = add_percent_columns(qc_summary)
flag_plot = flag_plot.melt(
    id_vars=["dataset", "source_type"],
    value_vars=["cells_lt_200_genes_pct", "cells_lt_500_genes_pct", "cells_mt_gt_10pct_pct", "cells_mt_gt_20pct_pct"],
    var_name="flag",
    value_name="percent_cells",
)
flag_plot["flag"] = flag_plot["flag"].map({
    "cells_lt_200_genes_pct": "<200 genes",
    "cells_lt_500_genes_pct": "<500 genes",
    "cells_mt_gt_10pct_pct": "MT >10%",
    "cells_mt_gt_20pct_pct": "MT >20%",
})

fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
for flag in flag_plot["flag"].dropna().unique():
    subset = flag_plot[flag_plot["flag"].eq(flag)].set_index("dataset").loc[order]
    ax.plot(order, subset["percent_cells"], marker="o", label=flag)
ax.set_ylabel("Cells flagged (%)")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=4, loc="upper left")
fig.savefig(OUT_DIR / "paper_dataset_qc_flags.png", dpi=220, bbox_inches="tight")
plt.show()

## Dataset summary

Summarize the computed QC statistics for the real datasets. Mitochondrial fractions are included where gene identifiers support prefix matching.


In [ ]:
def fmt_range(series: pd.Series, digits: int = 1) -> str:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return "not available"
    lo = values.min()
    hi = values.max()
    if digits == 0:
        return f"{lo:,.0f}-{hi:,.0f}"
    return f"{lo:,.{digits}f}-{hi:,.{digits}f}"


def draft_materials_paragraph(qc: pd.DataFrame, include_synthetic: bool = False) -> str:
    work = qc.copy()
    if not include_synthetic:
        work = work[work["source_type"].eq("real")].copy()

    n_matrices = len(work)
    n_cells = int(work["n_cells"].sum())
    n_genes_min = int(work["n_genes"].min())
    n_genes_max = int(work["n_genes"].max())
    zero_cell_total = int(work["zero_count_cells"].sum())
    zero_gene_cell_total = int(work["zero_detected_gene_cells"].sum())

    mt_available = work[work["mt_genes_detected"] > 0].copy()
    mt_note = "Mitochondrial genes were not identifiable by prefix in all source matrices."
    if not mt_available.empty:
        mt_note = (
            f"For matrices with mitochondrial gene identifiers, median mitochondrial fractions ranged from "
            f"{fmt_range(mt_available['pct_mt_median'], 2)}%, with 95th percentiles of "
            f"{fmt_range(mt_available['pct_mt_p95'], 2)}%."
        )
        mt20 = int(mt_available["cells_mt_gt_20pct"].fillna(0).sum())
        mt20_pct = mt20 / int(mt_available["n_cells"].sum()) * 100
        mt_note += f" Cells above 20% mitochondrial counts numbered {mt20:,} ({mt20_pct:.3f}% of cells)."

    empty_gene = work["zero_detected_cell_genes"].fillna(0)
    empty_gene_note = "No empty genes were present in the configured real matrices."
    if empty_gene.sum() > 0:
        affected = work.loc[empty_gene > 0, ["dataset", "zero_detected_cell_genes", "n_genes"]]
        affected_text = "; ".join(
            f"{row.dataset}: {int(row.zero_detected_cell_genes):,}/{int(row.n_genes):,} genes"
            for row in affected.itertuples(index=False)
        )
        empty_gene_note = (
            "Feature-level genes with no observed counts occurred in some condition-specific or nucleus matrices "
            f"({affected_text})."
        )

    paragraph = (
        f"We calculated QC metrics for the {n_matrices} real count matrices used in the analyses "
        f"({n_cells:,} cells; {n_genes_min:,}-{n_genes_max:,} genes per matrix). "
        f"Across these matrices, the totals for zero-count cells and cells with zero detected genes were "
        f"{zero_cell_total:,} and {zero_gene_cell_total:,}, respectively. "
        f"Median library sizes ranged from {fmt_range(work['counts_per_cell_median'], 0)} counts per cell, "
        f"and median detected genes ranged from {fmt_range(work['genes_per_cell_median'], 0)} genes per cell; "
        f"the corresponding 5th percentiles were {fmt_range(work['counts_per_cell_p05'], 0)} counts and "
        f"{fmt_range(work['genes_per_cell_p05'], 0)} genes. "
        f"{mt_note} {empty_gene_note} "
    )
    return paragraph

paragraph = draft_materials_paragraph(qc_summary, include_synthetic=False)
print(paragraph)

## Optional: Top Genes Per Dataset

These are useful for spotting expected high-abundance genes such as ribosomal, mitochondrial, hemoglobin, or MALAT1-like features.

In [ ]:
top_genes = (
    var_qc.sort_values(["dataset", "counts_per_gene"], ascending=[True, False])
    .groupby("dataset")
    .head(10)
    [["dataset", "gene", "counts_per_gene", "cells_per_gene", "pct_cells", "is_mt"]]
    .copy()
)

top_genes.to_csv(OUT_DIR / "paper_dataset_top_genes.csv", index=False)
top_genes

## Interpreting the metrics

- Mitochondrial fractions depend on identifiable mitochondrial gene prefixes.
- Genes with no observed counts and cells with no observed counts are reported separately.
- For simulated data, library sizes and detection rates describe the generated matrices; mitochondrial fractions have no biological interpretation.
